In [2]:
# ============================================================
# 03_frame_extraction.ipynb
# FAST FRAME EXTRACTION WITH FFMPEG + PARALLELISM
# ============================================================
# Features:
# 1. Uses FFmpeg for fast extraction
# 2. Parallel extraction across many videos
# 3. Tries GPU decode first if available
# 4. Falls back to CPU automatically
# 5. Saves frames into structured folders
# 6. Saves extraction summary CSV + JSON
#
# Output folders:
# data/processed/frames/RGB/<distance>/<gesture>/<subject_id>/<pair_id>/
# data/processed/frames/NIR/<distance>/<gesture>/<subject_id>/<pair_id>/
#
# Output files:
# data/processed/manifests/frame_extraction_summary.csv
# data/processed/manifests/frame_extraction_summary.json
# ============================================================

import os
import re
import cv2
import json
import math
import time
import shutil
import subprocess
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

# ============================================================
# 1. PATHS
# ============================================================
cwd = Path.cwd()

if cwd.name == "notebooks":
    ROOT = cwd.parent
else:
    ROOT = cwd

MANIFEST_PATH = ROOT / "data" / "processed" / "manifests" / "paired_manifest_with_subjects.csv"
FRAMES_ROOT = ROOT / "data" / "processed" / "frames"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"

FRAMES_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_summary.csv"
SUMMARY_JSON = MANIFEST_DIR / "frame_extraction_summary.json"

print("Current working directory :", cwd)
print("Project ROOT              :", ROOT)
print("Manifest path             :", MANIFEST_PATH)
print("Frames root               :", FRAMES_ROOT)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

# ============================================================
# 2. SETTINGS
# ============================================================
# For quick test set LIMIT_ROWS = 10, then later None
LIMIT_ROWS = 10

# Whether to skip extraction if target folder already has frames
SKIP_IF_ALREADY_EXTRACTED = True

# If True, delete old extracted folder and extract again
FORCE_REEXTRACT = False

# JPEG quality for FFmpeg (2 is very high quality, 31 is low quality)
# 2 to 4 is good
JPEG_QSCALE = 2

# Parallel workers
# For CPU fallback, 8-16 is often good depending on storage
# For GPU decode, too many workers may reduce efficiency
MAX_WORKERS = 8

# Try GPU decode first
TRY_GPU_DECODE = True

# Extract both modalities
EXTRACT_RGB = True
EXTRACT_NIR = True

# File extension for saved frames
FRAME_EXT = "jpg"

# Verbose subprocess output
VERBOSE = False

# ============================================================
# 3. LOAD MANIFEST
# ============================================================
df = pd.read_csv(MANIFEST_PATH)

required_cols = [
    "pair_id", "subject_id", "gesture", "distance",
    "rgb_video_path", "nir_video_path", "is_valid_pair"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in manifest: {missing_cols}")

df = df[df["is_valid_pair"] == True].copy().reset_index(drop=True)

if LIMIT_ROWS is not None:
    df = df.head(LIMIT_ROWS).copy()

print("\nRows to process:", len(df))
display(df.head())

# ============================================================
# 4. HELPERS
# ============================================================
def safe_str(x):
    if pd.isna(x):
        return "UNKNOWN"
    return str(x)

def make_output_dir(modality, distance, gesture, subject_id, pair_id):
    return FRAMES_ROOT / modality / safe_str(distance) / safe_str(gesture) / safe_str(subject_id) / safe_str(pair_id)

def count_existing_frames(folder: Path):
    if not folder.exists():
        return 0
    return len(list(folder.glob(f"frame_*.{FRAME_EXT}")))

def ffmpeg_exists():
    try:
        result = subprocess.run(
            ["ffmpeg", "-version"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        return result.returncode == 0
    except Exception:
        return False

def ffprobe_exists():
    try:
        result = subprocess.run(
            ["ffprobe", "-version"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        return result.returncode == 0
    except Exception:
        return False

if not ffmpeg_exists():
    raise RuntimeError("ffmpeg not found. Please install ffmpeg first.")

if not ffprobe_exists():
    raise RuntimeError("ffprobe not found. Please install ffprobe first.")

def get_hwaccels():
    try:
        result = subprocess.run(
            ["ffmpeg", "-hwaccels"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        return result.stdout.lower()
    except Exception:
        return ""

def get_decoders():
    try:
        result = subprocess.run(
            ["ffmpeg", "-decoders"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        return result.stdout.lower()
    except Exception:
        return ""

HWACCELS_TEXT = get_hwaccels()
DECODERS_TEXT = get_decoders()

HAS_CUDA_HWACCEL = "cuda" in HWACCELS_TEXT
HAS_CUVID = "cuvid" in DECODERS_TEXT

print("\nFFmpeg CUDA hwaccel available:", HAS_CUDA_HWACCEL)
print("FFmpeg CUVID decoder available:", HAS_CUVID)

def ffprobe_video_meta(video_path):
    """
    Use ffprobe for fast metadata extraction.
    """
    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height,r_frame_rate,avg_frame_rate,nb_frames",
        "-show_entries", "format=duration",
        "-of", "json",
        str(video_path)
    ]

    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.returncode != 0:
            return {
                "opened": False,
                "fps": np.nan,
                "frame_count_reported": np.nan,
                "width": np.nan,
                "height": np.nan,
                "duration_sec": np.nan,
            }

        info = json.loads(result.stdout)
        streams = info.get("streams", [])
        fmt = info.get("format", {})

        if not streams:
            return {
                "opened": False,
                "fps": np.nan,
                "frame_count_reported": np.nan,
                "width": np.nan,
                "height": np.nan,
                "duration_sec": np.nan,
            }

        s = streams[0]

        def parse_rate(rate_str):
            if not rate_str or rate_str == "0/0":
                return np.nan
            if "/" in rate_str:
                a, b = rate_str.split("/")
                a = float(a)
                b = float(b)
                return a / b if b != 0 else np.nan
            return float(rate_str)

        fps = parse_rate(s.get("avg_frame_rate", None))
        if np.isnan(fps):
            fps = parse_rate(s.get("r_frame_rate", None))

        nb_frames = s.get("nb_frames", None)
        try:
            nb_frames = int(nb_frames) if nb_frames is not None else np.nan
        except:
            nb_frames = np.nan

        duration = fmt.get("duration", None)
        try:
            duration = float(duration) if duration is not None else np.nan
        except:
            duration = np.nan

        width = s.get("width", np.nan)
        height = s.get("height", np.nan)

        return {
            "opened": True,
            "fps": float(fps) if not np.isnan(fps) else np.nan,
            "frame_count_reported": nb_frames,
            "width": int(width) if width is not None and not pd.isna(width) else np.nan,
            "height": int(height) if height is not None and not pd.isna(height) else np.nan,
            "duration_sec": float(duration) if not np.isnan(duration) else np.nan,
        }
    except Exception:
        return {
            "opened": False,
            "fps": np.nan,
            "frame_count_reported": np.nan,
            "width": np.nan,
            "height": np.nan,
            "duration_sec": np.nan,
        }

def build_ffmpeg_command(video_path, output_pattern, use_gpu_decode=True):
    """
    Build FFmpeg command.
    If GPU decode is available, try it first.
    Otherwise use CPU decode.
    """
    video_path = str(video_path)
    output_pattern = str(output_pattern)

    if use_gpu_decode and HAS_CUDA_HWACCEL:
        # Safe GPU-decode attempt
        # We do decode with CUDA if available, then write jpg frames.
        cmd = [
            "ffmpeg",
            "-y",
            "-hwaccel", "cuda",
            "-i", video_path,
            "-qscale:v", str(JPEG_QSCALE),
            output_pattern
        ]
    else:
        cmd = [
            "ffmpeg",
            "-y",
            "-i", video_path,
            "-qscale:v", str(JPEG_QSCALE),
            output_pattern
        ]

    return cmd

def run_ffmpeg_extract(video_path, output_dir, skip_if_done=True, force_reextract=False, try_gpu=True):
    """
    Extract all frames using FFmpeg.
    """
    output_dir = Path(output_dir)

    if force_reextract and output_dir.exists():
        shutil.rmtree(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    existing_count = count_existing_frames(output_dir)
    if skip_if_done and existing_count > 0:
        meta = ffprobe_video_meta(video_path)
        return {
            "status": "skipped_already_extracted",
            "frames_extracted": existing_count,
            "output_dir": str(output_dir),
            "used_gpu_decode": False,
            **meta
        }

    meta = ffprobe_video_meta(video_path)
    if not meta["opened"]:
        return {
            "status": "ffprobe_failed",
            "frames_extracted": 0,
            "output_dir": str(output_dir),
            "used_gpu_decode": False,
            **meta
        }

    output_pattern = output_dir / f"frame_%06d.{FRAME_EXT}"

    # First try GPU decode if requested
    used_gpu_decode = False
    if try_gpu and HAS_CUDA_HWACCEL:
        gpu_cmd = build_ffmpeg_command(video_path, output_pattern, use_gpu_decode=True)
        gpu_result = subprocess.run(
            gpu_cmd,
            stdout=subprocess.PIPE if not VERBOSE else None,
            stderr=subprocess.PIPE if not VERBOSE else None,
            text=True
        )

        if gpu_result.returncode == 0:
            extracted = count_existing_frames(output_dir)
            return {
                "status": "success_gpu",
                "frames_extracted": extracted,
                "output_dir": str(output_dir),
                "used_gpu_decode": True,
                **meta
            }
        else:
            # clean partial outputs before CPU retry
            for f in output_dir.glob(f"frame_*.{FRAME_EXT}"):
                try:
                    f.unlink()
                except:
                    pass

    # CPU fallback
    cpu_cmd = build_ffmpeg_command(video_path, output_pattern, use_gpu_decode=False)
    cpu_result = subprocess.run(
        cpu_cmd,
        stdout=subprocess.PIPE if not VERBOSE else None,
        stderr=subprocess.PIPE if not VERBOSE else None,
        text=True
    )

    if cpu_result.returncode == 0:
        extracted = count_existing_frames(output_dir)
        return {
            "status": "success_cpu",
            "frames_extracted": extracted,
            "output_dir": str(output_dir),
            "used_gpu_decode": False,
            **meta
        }

    return {
        "status": "ffmpeg_failed",
        "frames_extracted": count_existing_frames(output_dir),
        "output_dir": str(output_dir),
        "used_gpu_decode": False,
        **meta
    }

# ============================================================
# 5. BUILD TASK LIST
# ============================================================
tasks = []

for _, row in df.iterrows():
    pair_id = safe_str(row["pair_id"])
    subject_id = safe_str(row["subject_id"])
    gesture = safe_str(row["gesture"])
    distance = safe_str(row["distance"])

    if EXTRACT_RGB:
        tasks.append({
            "pair_id": pair_id,
            "subject_id": subject_id,
            "gesture": gesture,
            "distance": distance,
            "modality": "RGB",
            "video_path": Path(row["rgb_video_path"]),
            "output_dir": make_output_dir("RGB", distance, gesture, subject_id, pair_id),
        })

    if EXTRACT_NIR:
        tasks.append({
            "pair_id": pair_id,
            "subject_id": subject_id,
            "gesture": gesture,
            "distance": distance,
            "modality": "NIR",
            "video_path": Path(row["nir_video_path"]),
            "output_dir": make_output_dir("NIR", distance, gesture, subject_id, pair_id),
        })

print("\nTotal extraction tasks:", len(tasks))
display(pd.DataFrame(tasks).head())

# ============================================================
# 6. PARALLEL EXTRACTION
# ============================================================
def process_one_task(task):
    stats = run_ffmpeg_extract(
        video_path=task["video_path"],
        output_dir=task["output_dir"],
        skip_if_done=SKIP_IF_ALREADY_EXTRACTED,
        force_reextract=FORCE_REEXTRACT,
        try_gpu=TRY_GPU_DECODE
    )

    return {
        "pair_id": task["pair_id"],
        "subject_id": task["subject_id"],
        "gesture": task["gesture"],
        "distance": task["distance"],
        "modality": task["modality"],
        "video_path": str(task["video_path"]),
        "extracted_dir": stats["output_dir"],
        "status": stats["status"],
        "used_gpu_decode": stats["used_gpu_decode"],
        "frames_extracted": stats["frames_extracted"],
        "fps": stats["fps"],
        "frame_count_reported": stats["frame_count_reported"],
        "duration_sec": stats["duration_sec"],
        "width": stats["width"],
        "height": stats["height"],
    }

results = []
start_time = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_one_task, task) for task in tasks]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting frames"):
        results.append(future.result())

elapsed_sec = time.time() - start_time

# ============================================================
# 7. SUMMARY DATAFRAME
# ============================================================
summary_df = pd.DataFrame(results)

# Sort neatly
summary_df = summary_df.sort_values(["modality", "distance", "gesture", "subject_id", "pair_id"]).reset_index(drop=True)

print("\nExtraction summary shape:", summary_df.shape)
display(summary_df.head())

summary_df.to_csv(SUMMARY_CSV, index=False)

summary_json = {
    "total_input_pairs": int(len(df)),
    "total_tasks": int(len(tasks)),
    "rgb_tasks": int((summary_df["modality"] == "RGB").sum()) if len(summary_df) else 0,
    "nir_tasks": int((summary_df["modality"] == "NIR").sum()) if len(summary_df) else 0,
    "status_counts": summary_df["status"].value_counts().to_dict() if len(summary_df) else {},
    "gpu_decode_used_count": int(summary_df["used_gpu_decode"].sum()) if len(summary_df) else 0,
    "total_frames_rgb": int(summary_df[summary_df["modality"] == "RGB"]["frames_extracted"].sum()) if len(summary_df) else 0,
    "total_frames_nir": int(summary_df[summary_df["modality"] == "NIR"]["frames_extracted"].sum()) if len(summary_df) else 0,
    "avg_frames_rgb": float(summary_df[summary_df["modality"] == "RGB"]["frames_extracted"].mean()) if len(summary_df[summary_df["modality"] == "RGB"]) else 0.0,
    "avg_frames_nir": float(summary_df[summary_df["modality"] == "NIR"]["frames_extracted"].mean()) if len(summary_df[summary_df["modality"] == "NIR"]) else 0.0,
    "elapsed_seconds": round(float(elapsed_sec), 2),
    "elapsed_minutes": round(float(elapsed_sec / 60.0), 2),
    "summary_csv": str(SUMMARY_CSV),
}

with open(SUMMARY_JSON, "w") as f:
    json.dump(summary_json, f, indent=4)

# ============================================================
# 8. REPORT
# ============================================================
print("\nSaved frame extraction summary CSV to:")
print(SUMMARY_CSV)

print("\nSaved frame extraction summary JSON to:")
print(SUMMARY_JSON)

print("\n========== FINAL EXTRACTION SUMMARY ==========")
print(json.dumps(summary_json, indent=4))

print("\nStatus counts:")
display(summary_df["status"].value_counts())

print("\nGPU decode usage counts:")
display(summary_df["used_gpu_decode"].value_counts(dropna=False))

print("\nFrames extracted by modality:")
display(summary_df.groupby("modality")["frames_extracted"].sum())

print("\nAverage frames per video by modality:")
display(summary_df.groupby("modality")["frames_extracted"].mean().round(2))

print("\nSample extracted folders:")
display(summary_df[["pair_id", "modality", "extracted_dir", "frames_extracted", "status"]].head(10))

Current working directory : /data/Sajjan_Singh/gesture_recognition/notebooks
Project ROOT              : /data/Sajjan_Singh/gesture_recognition
Manifest path             : /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/paired_manifest_with_subjects.csv
Frames root               : /data/Sajjan_Singh/gesture_recognition/data/processed/frames

Rows to process: 10


,pair_id,pair_index_within_group,subject_id,gesture,distance,capture_id,rgb_video_path,nir_video_path,rgb_num_frames,nir_num_frames,rgb_fps,nir_fps,duration_rgb,duration_nir,fps_diff,frame_diff,duration_diff,is_valid_pair,rgb_width,rgb_height,nir_width,nir_height,rgb_filename,nir_filename,rgb_serial_no,nir_serial_no
0,PAIR_00001,1,4F_S001,doctor,4_feet,1,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_154701.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303154701830.mp4,184,135,29.696472,25.074294,6.196022,5.384,4.622178,49,0.812022,True,1080,1920,720,960,video_20260303_154701.mp4,20260303154701830.mp4,20260303154701,20260303154701830
1,PAIR_00002,2,4F_S002,doctor,4_feet,2,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_155209.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303155208958.mp4,173,137,30.019203,25.041126,5.762978,5.471,4.978077,36,0.291978,True,1080,1920,720,960,video_20260303_155209.mp4,20260303155208958.mp4,20260303155209,20260303155208958
2,PAIR_00003,3,4F_S003,doctor,4_feet,3,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_160149.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303160149432.mp4,180,137,30.019290,25.009127,5.996144,5.478,5.010163,43,0.518144,True,1080,1920,720,960,video_20260303_160149.mp4,20260303160149432.mp4,20260303160149,20260303160149432
3,PAIR_00004,4,4F_S004,doctor,4_feet,4,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161125.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161125449.mp4,175,142,30.019270,25.035261,5.829589,5.672,4.984009,33,0.157589,True,1080,1920,720,960,video_20260303_161125.mp4,20260303161125449.mp4,20260303161125,20260303161125449
4,PAIR_00005,5,4F_S005,doctor,4_feet,5,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161400.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161400573.mp4,167,135,30.019294,25.046382,5.563089,5.390,4.972912,32,0.173089,True,1080,1920,720,960,video_20260303_161400.mp4,20260303161400573.mp4,20260303161400,20260303161400573



FFmpeg CUDA hwaccel available: True
FFmpeg CUVID decoder available: True

Total extraction tasks: 20


,pair_id,subject_id,gesture,distance,modality,video_path,output_dir
0,PAIR_00001,4F_S001,doctor,4_feet,RGB,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_154701.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/RGB/4_feet/doctor/4F_S001/PAIR_00001
1,PAIR_00001,4F_S001,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303154701830.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S001/PAIR_00001
2,PAIR_00002,4F_S002,doctor,4_feet,RGB,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_155209.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/RGB/4_feet/doctor/4F_S002/PAIR_00002
3,PAIR_00002,4F_S002,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303155208958.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S002/PAIR_00002
4,PAIR_00003,4F_S003,doctor,4_feet,RGB,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_160149.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/RGB/4_feet/doctor/4F_S003/PAIR_00003


Extracting frames: 100%|██████████| 20/20 [00:14<00:00,  1.40it/s]

...

modality
NIR    1345
RGB    1747
Name: frames_extracted, dtype: int64


Average frames per video by modality:


modality
NIR    134.5
RGB    174.7
Name: frames_extracted, dtype: float64


Sample extracted folders:


,pair_id,modality,extracted_dir,frames_extracted,status
0,PAIR_00001,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S001/PAIR_00001,135,success_gpu
1,PAIR_00002,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S002/PAIR_00002,137,success_gpu
2,PAIR_00003,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S003/PAIR_00003,137,success_gpu
3,PAIR_00004,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S004/PAIR_00004,143,success_gpu
4,PAIR_00005,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S005/PAIR_00005,135,success_gpu
5,PAIR_00006,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S006/PAIR_00006,129,success_gpu
6,PAIR_00007,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S007/PAIR_00007,138,success_gpu
7,PAIR_00008,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S008/PAIR_00008,123,success_gpu
8,PAIR_00009,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S009/PAIR_00009,139,success_gpu
9,PAIR_00010,NIR,/data/Sajjan_Singh/gesture_recognition/data/processed/frames/NIR/4_feet/doctor/4F_S010/PAIR_00010,129,success_gpu
